In [5]:
#A life check scripts: what parts are live? how many messagesis kafka queue
import json
import sqlite3
import subprocess
from kafka import KafkaConsumer, TopicPartition
from kafka.errors import NoBrokersAvailable

with open("config.json", "r", encoding="utf-8") as f:
    config = json.load(f)

k_cfg = config.get("kafka") or config.get("kafka_topic", {})
BROKER = k_cfg.get("bootstrap_server", "localhost:9092")
TOPIC_RAW = k_cfg.get("raw_topic", "telegram-raw")
TOPIC_TOKENS = k_cfg.get("tokens_topic", "telegram-tokens")
CONSUMER_GROUP = "leaky-bucket-evaluator-group"

ps_output = subprocess.getoutput("ps aux")

# 1. Pipeline Processes Status
print("=== 1. WORKERS & LISTENERS ===")

# Live Telethon Ingestion
is_telethon = "tele_session" in ps_output
print(f"• Live Telethon Listener: {'[RUNNING]' if is_telethon else '[DEAD / STOPPED]'} (Notebook: producer.ipynb)")

# Stage 1 Worker (Spark Tokenizer)
is_spark = "org.apache.spark" in ps_output or "pyspark" in ps_output
print(f"• Stage 1 Worker (Spark): {'[RUNNING]' if is_spark else '[DEAD / STOPPED]'} (Notebook: sparkRawToTokens.ipynb)")

# Stage 2 Worker (Tokens Counter)
is_counter = "sparkTokensCounter" in ps_output or CONSUMER_GROUP in ps_output
print(f"• Stage 2 Worker:         {'[RUNNING]' if is_counter else '[DEAD / STOPPED]'} (Notebook: sparkTokensCounter.ipynb)")

# 2. Kafka Queues Status
print("\n=== 2. KAFKA QUEUES ===")
try:
    consumer = KafkaConsumer(
        bootstrap_servers=BROKER,
        request_timeout_ms=15000,
        session_timeout_ms=10000,
        enable_auto_commit=False
    )
    topics = consumer.topics()

    # First Queue (telegram-raw)
    if TOPIC_RAW in topics:
        parts_raw = consumer.partitions_for_topic(TOPIC_RAW) or []
        tps_raw = [TopicPartition(TOPIC_RAW, p) for p in parts_raw]
        total_raw = sum(consumer.end_offsets(tps_raw).values())
        print(f"• First Queue ('{TOPIC_RAW}'): [READY] | Total Messages: {total_raw:,}")
    else:
        print(f"• First Queue ('{TOPIC_RAW}'): [DEAD / NOT CREATED]")

    # Second Queue (telegram-tokens)
    if TOPIC_TOKENS in topics:
        parts_tokens = consumer.partitions_for_topic(TOPIC_TOKENS) or []
        tps_tokens = [TopicPartition(TOPIC_TOKENS, p) for p in parts_tokens]
        end_offs = consumer.end_offsets(tps_tokens)
        total_tokens = sum(end_offs.values())

        # Measure pending backlog in second queue
        total_lag = 0
        try:
            group_consumer = KafkaConsumer(
                bootstrap_servers=BROKER,
                group_id=CONSUMER_GROUP,
                enable_auto_commit=False,
                request_timeout_ms=15000,
                session_timeout_ms=10000
            )
            for tp in tps_tokens:
                comm = group_consumer.committed(tp) or 0
                total_lag += max(0, end_offs[tp] - comm)
            group_consumer.close()
        except Exception:
            total_lag = 0

        queue_state = "EMPTY (0 unread)" if total_lag == 0 else f"HAS BACKLOG ({total_lag:,} unread messages)"
        print(f"• Second Queue ('{TOPIC_TOKENS}'): [READY] | Total Processed: {total_tokens:,} | Queue Status: {queue_state}")
    else:
        print(f"• Second Queue ('{TOPIC_TOKENS}'): [DEAD / NOT CREATED]")

    consumer.close()

except NoBrokersAvailable:
    print("• Kafka Server: [DEAD / STOPPED] - Run 'bash run_kafka.sh' in terminal")
except Exception as e:
    print(f"• Kafka Error: {e}")

=== 1. WORKERS & LISTENERS ===
• Live Telethon Listener: [DEAD / STOPPED] (Notebook: producer.ipynb)
• Stage 1 Worker (Spark): [DEAD / STOPPED] (Notebook: sparkRawToTokens.ipynb)
• Stage 2 Worker:         [DEAD / STOPPED] (Notebook: sparkTokensCounter.ipynb)

=== 2. KAFKA QUEUES ===
• First Queue ('telegram-raw'): [DEAD / NOT CREATED]
• Second Queue ('telegram-tokens'): [DEAD / NOT CREATED]


In [4]:
#A POWERFULL SCRIPT TO KILL ALL CURRENT PROCESS
import os
import json
import shutil
import subprocess
from kafka import KafkaAdminClient
from kafka.errors import NoBrokersAvailable

print("Stopping background processes and clearing queues...")

subprocess.run(["pkill", "-f", "org.apache.spark"])
subprocess.run(["pkill", "-f", "pyspark"])
subprocess.run(["pkill", "-f", "tele_session"])
print("Terminated Spark and Telethon workers.")

checkpoint_dir = "/tmp/spark-kafka-tokenizer-checkpoint"
if os.path.exists(checkpoint_dir):
    shutil.rmtree(checkpoint_dir, ignore_errors=True)
    print(f"Cleared Spark checkpoint directory: {checkpoint_dir}")

try:
    with open("config.json", "r", encoding="utf-8") as f:
        config = json.load(f)
    
    k_cfg = config.get("kafka") or config.get("kafka_topic", {})
    broker = k_cfg.get("bootstrap_server", "localhost:9092")
    raw_topic = k_cfg.get("raw_topic", "telegram-raw")
    tokens_topic = k_cfg.get("tokens_topic", "telegram-tokens")

    admin = KafkaAdminClient(bootstrap_servers=broker, request_timeout_ms=5000)
    topics_to_delete = [t for t in [raw_topic, tokens_topic] if t in admin.list_topics()]
    
    if topics_to_delete:
        admin.delete_topics(topics_to_delete)
        print(f"Deleted topics to clear queues: {topics_to_delete}")
    else:
        print("Topics were already clean.")
    admin.close()
except NoBrokersAvailable:
    print("Kafka broker is offline; skipped queue purge.")
except Exception as e:
    print(f"Kafka cleanup note: {e}")

print("System clean. Click 'Restart Kernel' in any open notebooks.")

Stopping background processes and clearing queues...
Terminated Spark and Telethon workers.
Cleared Spark checkpoint directory: /tmp/spark-kafka-tokenizer-checkpoint
Deleted topics to clear queues: ['telegram-raw', 'telegram-tokens']
System clean. Click 'Restart Kernel' in any open notebooks.
